# RF Testing

Checking path so that I know which qickdawg it's using. It should be the one in the local folder and not the github 

In [1]:
import sys
print(sys.path)

['C:\\Program Files\\Swabian Instruments\\Time Tagger\\driver\\python', 'c:\\msys64\\clang64\\lib\\python312.zip', 'c:\\msys64\\clang64\\lib\\python3.12', 'c:\\msys64\\clang64\\lib\\python3.12\\lib-dynload', '', 'c:\\msys64\\clang64\\lib\\python3.12\\site-packages', 'C:\\Users\\DQP\\repos\\dqpcontrol\\qick-dawg\\src', '__editable__.rf_awg-0.1.0.finder.__path_hook__', 'c:\\msys64\\clang64\\lib\\python3.12\\site-packages\\win32', 'c:\\msys64\\clang64\\lib\\python3.12\\site-packages\\win32\\lib', 'c:\\msys64\\clang64\\lib\\python3.12\\site-packages\\Pythonwin']


In [1]:
!pip show qickdawg

Name: qickdawg
Version: 1.2.1
Summary: Software for full quantum control of nitrogen-vacancy defects and other quantum defects in diamond
Home-page: 
Author: 
Author-email: Andy Mounce <amounce@sandia.gov>, Emmeline Riendeau <eriendeau@uchicago.edu>
License: MIT License 

Copyright 2023 National Technology & Engineering Solutions of Sandia, LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the U.S. Government retains certain rights in this software.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substa

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

c:\msys64\clang64\lib\python3.12\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
qd.start_client('192.168.0.100')

QICK library version mismatch: 0.2.324 remote (the board), 0.2.302 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


## Load Default Configuration

In [4]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.edge_counting = False
# default_config.high_threshold = 2000
# default_config.low_threshold = 500

default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 100

# default_config.laser_gate_pmod = 0

# default_config.relax_delay_tns = 50 # between each rep, wait for everything to catch up, mostly aom


# CPMGXY8


In [39]:
from qickdawg.arqick.legacy.arqick_cpmg_XY8 import CPMGXY8

soc = qd.soc
config = copy(default_config)
config.mw_gain = 32000
config.mw_pi2_tns = 50
config.freq_fMHz = 500 # in Hz
config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=50)
config.n_cpmg = 1 # number of cpmg xy8 rounds
config.pulse_seq_delay_tus = 1
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 0

prog = CPMGXY8(config)
prog.run_rounds(soc, rounds=1, start_src='external')


Requested 10 to 100 by 50
Instead using 9.765625 to 58.59375 by 48.828125 in 2 steps


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:02<00:00,  1.48s/it]


# CPMG (phase = 0)


In [6]:
from qickdawg.arqick.arqick_cpmg import CPMG

soc = qd.soc
config = copy(default_config)
config.mw_gain = 30000
config.mw_pi2_tns = 100
config.freq_fMHz = 500 # in Hz
config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=50)
config.n_cpmg = 10 # number of pi pulses in the cpmg sequence
config.pulse_seq_delay_tus = 1.5
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = CPMG(config)
prog.run_rounds(soc, rounds=1)

ModuleNotFoundError: No module named 'qickdawg.arqick.arqick_cpmg'

# ramsey

In [308]:
from qickdawg.arqick.arqick_ramsey import Ramsey

soc = qd.soc
config = copy(default_config)
config.mw_gain = 30000
config.mw_pi2_tns = 100
config.freq_fMHz = 500 # in Hz
config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=50)
config.pulse_seq_delay_tus = 1.5
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = Ramsey(config)
prog.run_rounds(soc, rounds=1)


Requested 10 to 100 by 50
Instead using 9.765625 to 58.59375 by 48.828125 in 2 steps


100%|██████████| 2/2 [00:00<?, ?it/s]


# Optimal Counting duration

In [303]:
from qickdawg.arqick.arqick_counting_duration import CountingDuration

soc = qd.soc
config = copy(default_config)
config.mw_gain = 30000
config.mw_duration_tns = 100
config.freq_fMHz = 500
config.pulse_seq_delay_tus = 1.5
config.reps=3
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = CountingDuration(config)
prog.run_rounds(soc, rounds=1)

100%|██████████| 3/3 [00:00<00:00, 2997.36it/s]


# PODMr

In [ ]:
from qickdawg.arqick.arqick_podmr import PODMR

soc = qd.soc
config = copy(default_config)
config.counting_duration_tns = 500
config.mw_gain = 30000
config.mw_duration_tns = 100
config.add_linear_sweep(name = "freq", unit = "fMHz", start = 400, stop = 500, delta=100)
config.pulse_seq_delay_tus = 1.5
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 100 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = PODMR(config)
prog.run_rounds(soc, rounds=1)


Requested 400 to 500 by 100
Instead using 399.99999961853024 to 499.99999923706054 by 99.99999961853027 in 2 steps


100%|██████████| 2/2 [00:00<00:00, 4606.59it/s]


# Arqick rabi

In [ ]:
from qickdawg.arqick.arqick_rabi import Rabi

soc = qd.soc
config = copy(default_config)
config.counting_duration_tns = 500
config.mw_gain = 30000
config.freq_fMHz = 500
config.add_linear_sweep(name = "mw_duration", unit = "tns", start = 10, stop = 100, delta=50)
print(config.mw_duration_start_tns)
config.pulse_seq_delay_tus = 1.5
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 100 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = Rabi(config)
prog.run_rounds(soc, rounds=1)


Requested 10 to 100 by 50
Instead using 9.765625 to 58.59375 by 48.828125 in 2 steps
9.765625


100%|██████████| 2/2 [00:00<00:00, 995.09it/s]


# Arick CWODMr

In [ ]:
from qickdawg.arqick.arqick_cwodmr import CWODMR

soc = qd.soc
config = copy(default_config)
config.counting_duration_tns = 480
config.mw_gain = 30000
config.freq_start_fMHz = 498
config.freq_end_fMHz = 500
config.nsweep_points = 4 # >0 or division by 0 error 
config.pulse_seq_delay_tus =105.48
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 5000 - 198
config.inherent_trigger_to_pulses_delay_tns = 198


prog = CWODMR(config)
prog.run_rounds(soc, rounds=0, start_src="external")


In [6]:
from qickdawg.nvpulsing.rfpulse import RFPulse

soc = qd.soc
config = copy(default_config)

config.readout_integration_tus = qd.max_int_time_tus # necessary but not used
config.mw_fMHz = 200# frequency; fMHz, freg, fGHz
config.mw_off_fMHz = 200 # off resonant frequency
config.pulse_len_tus = 50 # pulse duration; tus, tns, treg
config.relax_delay_tns = 20 # 1.62 us
config.gain = 32767 # up to 32767 
config.reps = 1000
config.num_pulses = 1
config.pmod_out_pin = 0 # PMOD0_0
config.pmod_out_pulse_width_tns = 100 # 10 treg = 80
config.pmod_out_trig_delay_tns = 0

config.init_delay_tns = 0# wait before starting the sequence
prog = RFPulse(config)
prog.run_rounds(soc,rounds = 1000)

100%|██████████| 1000/1000 [00:54<00:00, 18.48it/s]


# CPMGXY8 
Simple CPMGXY for investigating looping in body and init

In [84]:
from qickdawg.nvpulsing.cpmgxy_test import CPMGXY8nDelaySweepInBody

soc = qd.soc
config = copy(default_config)

config.mw_gain = 30000
config.mw_fMHz = 500
config.mw_pi2_tns = 100

config.relax_delay_tus = 0.05

config.scaling_mode = 'linear'
config.delay_start_tns = 50
config.delay_end_tns = 100
config.nsweep_points = 1 # >0 or division by 0 error 
config.pmod_out_pin = 0 # PMOD0_0
config.pmod_out_pulse_width_tns = 10 # 10 treg = 80
config.pmod_out_trig_delay_tns = 0

config.reps=1
config.n_cpmg = 1
prog = CPMGXY8nDelaySweepInBody(config)
prog.run_rounds(soc, rounds=0, start_src="external")

In [72]:
delays = np.linspace(50, 100, 2)
print(delays.size)

2


# CPMGXY8 Sweep Param outside

In [ ]:
from qickdawg.nvpulsing.cpmgxy_sweep_outside import CPMGXY8nDelaySweepOutside

delays = np.linspace(50, 100, 2)
for i in range(delays.size):
    soc = qd.soc
    config = copy(default_config)

    config.mw_gain = 30000
    config.mw_fMHz = 500
    config.mw_pi2_tns = 100

    config.relax_delay_tus = 0.5
    config.delay_tns = delays[i]

    config.reps=1
    config.n_cpmg = 1
    prog = CPMGXY8nDelaySweepOutside(config)
    prog.run_rounds(soc, rounds=1, start_src="external")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:01<00:00,  1.44it/s]


# Gain Sweep (viewable on oscilliscope)

pmod channel width not tunable when time is very short it seems.

In [24]:
from qickdawg.nvpulsing.rftest import RFTest 

config = copy(default_config)

config.readout_integration_tus = qd.max_int_time_tus # necessary but not used
config.mw_fMHz = 1410 # frequency; fMHz, freg, fGHz
config.pulse_len_tus = 5000 # pulse duration; tus, tns, treg
config.relax_delay_tns = 20 # 1.62 us
config.trigger_width_tns = 100

config.add_unitless_linear_sweep("gain", 100, 1000, delta=10)
config.pre_init = False # not sure if necessary
config.reps = 1

config.repitition = 1

prog = RFTest(config)
_ = prog.acquire()

RuntimeError: Pulse length of 1536000 is out of range (exceeds 16 bits, or less than 3) - use multiple pulses, or zero-pad the waveform

## Envelopes!

In [12]:
# Get the maxv for envelopes on the mw_channel
default_config.soccfg.get_maxv(default_config.mw_channel)
# maxv is tied to gain units
# if maxv isn't declared then the envelope will use the maximum as default

32766

In [10]:
from qickdawg.testfunctions.rftest_envelope import RFTest_Envelope

config = copy(default_config)

config.readout_integration_tus = qd.max_int_time_tus # necessary but not used
config.mw_fMHz = 2700 # frequency; fMHz, freg, fGHz
config.pulse_len_tns = 300 # pulse duration; tus, tns, treg
config.pulse_sigma_tns = 100 # sigma for gaussian pulse
config.relax_delay_tns = 500 # 1.62 us
config.trigger_width_tns = 500

config.add_unitless_linear_sweep("gain", 32000, 31900, delta=-1)
config.pre_init = False # not sure if necessary
config.reps = 1
config.repitition = 1

prog = RFTest_Envelope(config)
_ = prog.acquire()


Requested 32000 to 31900 by -1
Instead using 32000 to 31899 by -1 in 101


## QICK ADC

In [22]:
from qickdawg.arqick.arqick_qick_adc import QickAdcTesting
# use artiq TTL56Pulse class (ttl56_pulse.py)
soc = qd.soc
config = copy(default_config)

config.edge_counting = False
config.pmod_out_pin = 0
config.pmod_out_pulse_width_tns = 300
config.adc_channel = 0
config.adc_trig_offset_tns = 0
config.relax_delay_tns = 200
config.readout_integration_tns = 1000
config.readout_threshold = 3000
config.mw_channel = 0
config.mw_fMHz = 200                
config.mw_pulse_len_tns = 100           
config.reps = 1
config.gain = 8000 

prog = QickAdcTesting(config)
data = prog.acquire_decimated(soc, start_src="external", progress=False)
print(data)
t_ns = 1/0.3072
time_axis = np.arange(len(data)) * t_ns
plt.figure(figsize=(10,4))
plt.plot(time_axis, data)
plt.title("ADC readout- Looking for TTL")
plt.xlabel("t (ns)")
plt.ylabel("ADC Units")
plt.legend()
plt.show()

KeyboardInterrupt: 

## ADC + RABI

In [85]:
from qickdawg.arqick.arqick_rabi_mbi_200ps import RabiMbiFineRes

soc = qd.soc
config = copy(default_config)

config.edge_counting = False
config.mw_duration_tdds_start = 500
config.mw_duration_tdds_end = 1000
config.nsweep_points = 5
config.freq_fMHz = 200                
config.mw_channel = 0
config.mw_nqz = 1
config.mw_gain = 8000 
config.reps = 1
config.pmod_out_pin = 0
config.pmod_out_pulse_width_tns = 300
config.pmod_out_trig_delay_tus = 5 # 62.136 | before delay
config.inherent_trigger_to_pulses_delay_tns = 209.27
config.pulse_seq_delay_tus = 15.8 # after delay
config.adc_channel = 0
config.readout_threshold = 8000
config.readout_integration_tns = 500
config.delay_before_readout_repeats_tns = 1000

prog = RabiMbiFineRes(config)
# prog.acquire(soc, start_src="external")
data = prog.acquire_decimated(soc, start_src="external", progress=False)
print(data)
t_ns = 1/0.3072
time_axis = np.arange(len(data)) * t_ns
plt.figure(figsize=(10,4))
plt.plot(time_axis, data)
plt.title("ADC readout- Looking for TTL")
plt.xlabel("t (ns)")
plt.ylabel("ADC Units")
plt.legend()
plt.show()

AssertionError: Missing value for config.delay_after_readout_window_to_mw_treg, config.extra_delay_treg, config.t_buffer_treg, config.qick_processing_time_after_readout_treg, config.delay_after_first_pmod_out_treg

## Regular red rabi

In [76]:
from qickdawg.arqick.arqick_rabi_200ps import RabiFineRes

soc = qd.soc
config = copy(default_config)

config.edge_counting = False
config.mw_duration_tdds_start = 500
config.mw_duration_tdds_end = 1000
config.nsweep_points = 5
config.freq_fMHz = 200                
config.mw_channel = 0
config.mw_nqz = 1
config.mw_gain = 8000 
config.reps = 1
config.pmod_out_pin = 0
config.pmod_out_pulse_width_tns = 300 # 50
config.pmod_out_trig_delay_tus = 5 # 62.136
# round((self.t_buffer_us*1e6 + self.green_to_red1*1e6 + self.green_init_duration*1e6 
# + self.a1_optical_pump*1e6 + self.after_red_op_to_mw_buffer*1e6 
# - config.inherent_trigger_to_pulses_delay_tns/1000),5) # before delay |the delay before qick pulses | to account for inherent delay for mw pulse to come out
# 50000 + 345 + 5000 + 5000 + 2000 - 209 = 62136 ns
config.inherent_trigger_to_pulses_delay_tns = 209.27
config.pulse_seq_delay_tus = 15.8 
#round(self.after_mw_to_spin_readout_buffer*1e6 + self.read_to_green*1e6 + 
#self.ex_spin_readout*1e6 + self.wait_time*1e6 + self.charge_readout*1e6 + 
#self.qick_experiment_padding*1e6,5) # after delay | the delay after qick pulses
# 100 + 190 + 10000 + 500 + 4000 + 1000 = 15790 ns = 15.79 us
prog = RabiFineRes(config)
prog.run_rounds(soc, rounds=1, start_src='external')

100%|██████████| 5/5 [00:02<00:00,  1.68it/s]
